# 05 - Prescription product neighbourhood summaries reported in the Results

This notebook creates readable prescription-drug Category 3 neighborhood tables from the selected Hidalgo product-space network. Nodes are Category 3 products; Category 2 labels use `Category 2 (most common)`.


In [ ]:
# --- config ---
HIDALGO_RCA_INPUT = "counts"  # "counts" | "binary"
HIDALGO_GRAPH = "plot_graph"  # "full_phi" | "plot_graph" | "disparity_backbone"
NEIGHBORS_PER_PRODUCT = 5
RX_PRODUCT_SCOPE = "all"  # "all" | "top_by_reviews" | "top_by_vendors"
TOP_RX_PRODUCTS = 20
EXCLUDE_RX_PRODUCTS = {"Custom Listing"}
MIN_PHI = 0.0

import os
import sys
import importlib
from pathlib import Path
from collections import Counter

import networkx as nx
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
import dnm_viz_utils as dvu
dvu = importlib.reload(dvu)

repo_root = dvu.repo_root
load_master_drugs = dvu.load_master_drugs
load_category2_most_common_mapping = dvu.load_category2_most_common_mapping
MARKETS = dvu.MARKETS
MARKET_FILE_NAMES = dvu.MARKET_FILE_NAMES

os.chdir(repo_root())
ROOT = repo_root()
OUT = ROOT / "prescription_reports/new_category2_networks"
TAB = OUT / "tables"
NET = OUT / "networks"
TAB.mkdir(parents=True, exist_ok=True)

def hidalgo_suffix() -> str:
    return "" if HIDALGO_RCA_INPUT == "counts" else f"_{HIDALGO_RCA_INPUT}"

def output_prefix() -> str:
    return f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_rx_neighborhood"

PREFIX = output_prefix()
print("Network:", PREFIX)


## Load Data and Selected Hidalgo Network

The network source is controlled by `HIDALGO_RCA_INPUT` and `HIDALGO_GRAPH`. Use `plot_graph` if you want these tables to match the figure/regression network created in notebook 02. Use `full_phi` if you want neighborhoods from the complete positive-phi product space.


In [ ]:
def clean_product(x) -> str:
    return str(x).strip()


def add_listing_id(df: pd.DataFrame) -> pd.Series:
    pages = df["Pages"] if "Pages" in df.columns else pd.Series(index=df.index, dtype=object)
    names = df["Productname"] if "Productname" in df.columns else pd.Series(index=df.index, dtype=object)
    valid_pages = pages.notna() & (pages.astype(str).str.strip() != "") & (pages.astype(str).str.lower() != "nan")
    return pages.where(valid_pages, names).astype(str).str.strip()


def product_support_table(df_drugs: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame:
    d = df_drugs.dropna(subset=["Website", "Category 3"]).copy()
    d["product_cat3"] = d["Category 3"].map(clean_product)
    d["vendor_id"] = d["ID_Num"].astype(str).str.strip()
    d["_listing_id"] = add_listing_id(d)
    d = d[(d["product_cat3"] != "") & (d["_listing_id"] != "")]
    out = (
        d.groupby(["Website", "product_cat3"], sort=False)
        .agg(
            reviews=("product_cat3", "size"),
            n_vendors=("vendor_id", lambda s: s.replace("nan", np.nan).dropna().nunique()),
            n_listings=("_listing_id", "nunique"),
        )
        .reset_index()
        .rename(columns={"Website": "market"})
    )
    out["category2"] = out["product_cat3"].map(lambda p: prod_to_cat2.get(p, "Unknown"))
    out["is_prescription"] = out["category2"].eq("Prescription")
    return out


def load_phi_edges_from_csv(tag: str) -> pd.DataFrame:
    p = TAB / f"{tag}_hidalgo{hidalgo_suffix()}_phi_edgelist.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run notebook 02 with matching HIDALGO_RCA_INPUT first.")
    edges = pd.read_csv(p)
    edges = edges.rename(columns={"product_i": "source", "product_j": "target"})
    edges["source"] = edges["source"].map(clean_product)
    edges["target"] = edges["target"].map(clean_product)
    edges["phi"] = edges["phi"].astype(float)
    return edges[["source", "target", "phi"]]


def load_edges_from_graphml(tag: str) -> pd.DataFrame:
    p = NET / f"{tag}_hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}.graphml"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run notebook 02 with matching settings first.")
    G = nx.read_graphml(p)
    rows = []
    for u, v, data in G.edges(data=True):
        phi = float(data.get("phi", data.get("weight", 0.0)))
        rows.append({"source": clean_product(u), "target": clean_product(v), "phi": phi})
    return pd.DataFrame(rows)


def load_selected_hidalgo_edges(tag: str) -> pd.DataFrame:
    if HIDALGO_GRAPH == "full_phi":
        edges = load_phi_edges_from_csv(tag)
    elif HIDALGO_GRAPH in {"plot_graph", "disparity_backbone"}:
        edges = load_edges_from_graphml(tag)
    else:
        raise ValueError("HIDALGO_GRAPH must be 'full_phi', 'plot_graph', or 'disparity_backbone'")
    return edges[edges["phi"] > MIN_PHI].copy()


df_drugs = load_master_drugs()
df_drugs = df_drugs[df_drugs["Website"].isin(MARKETS)].copy()
prod_to_cat2 = load_category2_most_common_mapping(df_drugs)
support = product_support_table(df_drugs, prod_to_cat2)
print(f"Loaded {len(df_drugs):,} drug rows; {len(prod_to_cat2):,} Category 3 mappings")
display(support[~support["product_cat3"].isin(EXCLUDE_RX_PRODUCTS)].head())


## Build Prescription Neighborhood Tables


In [ ]:
def choose_rx_products(market: str, support_market: pd.DataFrame, edge_nodes: set[str]) -> list[str]:
    rx = support_market[
        support_market["is_prescription"]
        & support_market["product_cat3"].isin(edge_nodes)
        & ~support_market["product_cat3"].isin(EXCLUDE_RX_PRODUCTS)
    ].copy()
    if RX_PRODUCT_SCOPE == "all":
        return sorted(rx["product_cat3"].unique())
    if RX_PRODUCT_SCOPE == "top_by_reviews":
        return rx.sort_values(["reviews", "product_cat3"], ascending=[False, True]).head(TOP_RX_PRODUCTS)["product_cat3"].tolist()
    if RX_PRODUCT_SCOPE == "top_by_vendors":
        return rx.sort_values(["n_vendors", "reviews", "product_cat3"], ascending=[False, False, True]).head(TOP_RX_PRODUCTS)["product_cat3"].tolist()
    raise ValueError("RX_PRODUCT_SCOPE must be 'all', 'top_by_reviews', or 'top_by_vendors'")


def support_lookup(support_market: pd.DataFrame, product: str, column: str, default=0):
    hit = support_market.loc[support_market["product_cat3"].eq(product), column]
    if hit.empty:
        return default
    return hit.iloc[0]


def build_market_neighborhoods(market: str, edges: pd.DataFrame, support_market: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    edge_nodes = set(edges["source"]).union(edges["target"])
    rx_products = choose_rx_products(market, support_market, edge_nodes)
    rows = []

    for rx_product in rx_products:
        touched = edges[(edges["source"].eq(rx_product)) | (edges["target"].eq(rx_product))].copy()
        if touched.empty:
            continue
        touched["neighbor_product"] = np.where(touched["source"].eq(rx_product), touched["target"], touched["source"])
        touched = touched[~touched["neighbor_product"].isin(EXCLUDE_RX_PRODUCTS)].copy()
        touched = touched.sort_values(["phi", "neighbor_product"], ascending=[False, True]).drop_duplicates("neighbor_product")
        for rank, row in enumerate(touched.head(NEIGHBORS_PER_PRODUCT).itertuples(index=False), start=1):
            neighbor = clean_product(row.neighbor_product)
            neighbor_category2 = prod_to_cat2.get(neighbor, "Unknown")
            rows.append({
                "market": market,
                "network_label": PREFIX.replace("_rx_neighborhood", ""),
                "rx_product": rx_product,
                "rx_reviews": int(support_lookup(support_market, rx_product, "reviews", 0)),
                "rx_n_vendors": int(support_lookup(support_market, rx_product, "n_vendors", 0)),
                "rx_n_listings": int(support_lookup(support_market, rx_product, "n_listings", 0)),
                "neighbor_rank": rank,
                "neighbor_product": neighbor,
                "neighbor_category2": neighbor_category2,
                "neighbor_is_prescription": bool(neighbor_category2 == "Prescription"),
                "neighbor_reviews": int(support_lookup(support_market, neighbor, "reviews", 0)),
                "neighbor_n_vendors": int(support_lookup(support_market, neighbor, "n_vendors", 0)),
                "neighbor_n_listings": int(support_lookup(support_market, neighbor, "n_listings", 0)),
                "phi": float(row.phi),
                "tie_type": "prescription-prescription" if neighbor_category2 == "Prescription" else "prescription-non-prescription",
            })

    ties = pd.DataFrame(rows)
    if ties.empty:
        return ties, pd.DataFrame(), pd.DataFrame()

    product_summary = (
        ties.groupby(["market", "rx_product"], as_index=False)
        .agg(
            rx_reviews=("rx_reviews", "first"),
            rx_n_vendors=("rx_n_vendors", "first"),
            rx_n_listings=("rx_n_listings", "first"),
            n_neighbors=("neighbor_product", "count"),
            mean_phi=("phi", "mean"),
            max_phi=("phi", "max"),
            n_prescription_neighbors=("neighbor_is_prescription", "sum"),
        )
    )
    product_summary["share_prescription_neighbors"] = product_summary["n_prescription_neighbors"] / product_summary["n_neighbors"]
    product_summary = product_summary.sort_values(["market", "rx_reviews", "rx_product"], ascending=[True, False, True])

    category_summary = (
        ties.groupby(["market", "neighbor_category2"], as_index=False)
        .agg(
            n_top_neighbor_ties=("phi", "count"),
            mean_phi=("phi", "mean"),
            n_distinct_rx_products=("rx_product", "nunique"),
            n_distinct_neighbor_products=("neighbor_product", "nunique"),
        )
    )
    category_summary["share_of_top_neighbor_ties"] = category_summary["n_top_neighbor_ties"] / category_summary.groupby("market")["n_top_neighbor_ties"].transform("sum")
    category_summary = category_summary.sort_values(["market", "n_top_neighbor_ties", "mean_phi"], ascending=[True, False, False])
    return ties, product_summary, category_summary


def build_full_edge_nonrx_hubs(market: str, edges: pd.DataFrame, support_market: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    rows = []
    for row in edges.itertuples(index=False):
        u, v, phi = clean_product(row.source), clean_product(row.target), float(row.phi)
        cu, cv = prod_to_cat2.get(u, "Unknown"), prod_to_cat2.get(v, "Unknown")
        if (cu == "Prescription") ^ (cv == "Prescription"):
            rx_product, neighbor = (u, v) if cu == "Prescription" else (v, u)
            if rx_product in EXCLUDE_RX_PRODUCTS:
                continue
            rows.append({
                "market": market,
                "nonrx_product": neighbor,
                "nonrx_category2": prod_to_cat2.get(neighbor, "Unknown"),
                "n_rx_nonrx_edges": 1,
                "mean_phi": phi,
            })
    if not rows:
        return pd.DataFrame()
    hubs = pd.DataFrame(rows)
    hubs = (
        hubs.groupby(["market", "nonrx_product", "nonrx_category2"], as_index=False)
        .agg(n_rx_nonrx_edges=("n_rx_nonrx_edges", "sum"), mean_phi=("mean_phi", "mean"))
        .sort_values(["market", "n_rx_nonrx_edges", "mean_phi", "nonrx_product"], ascending=[True, False, False, True])
    )
    hubs["nonrx_reviews"] = hubs["nonrx_product"].map(lambda p: int(support_lookup(support_market, p, "reviews", 0)))
    hubs["nonrx_n_vendors"] = hubs["nonrx_product"].map(lambda p: int(support_lookup(support_market, p, "n_vendors", 0)))
    return hubs.groupby("market", as_index=False).head(top_n)


all_ties = []
all_product_summaries = []
all_category_summaries = []
all_hubs = []
market_rows = []

for market in MARKETS:
    tag = MARKET_FILE_NAMES[market]
    edges = load_selected_hidalgo_edges(tag)
    support_market = support[support["market"].eq(market)].copy()
    ties, product_summary, category_summary = build_market_neighborhoods(market, edges, support_market)
    hubs = build_full_edge_nonrx_hubs(market, edges, support_market)

    all_ties.append(ties)
    all_product_summaries.append(product_summary)
    all_category_summaries.append(category_summary)
    all_hubs.append(hubs)

    n_ties = len(ties)
    market_rows.append({
        "market": market,
        "network_label": PREFIX.replace("_rx_neighborhood", ""),
        "rx_products_in_table": ties["rx_product"].nunique() if not ties.empty else 0,
        "top_neighbor_ties": n_ties,
        "share_prescription_prescription": float((ties["tie_type"].eq("prescription-prescription")).mean()) if n_ties else np.nan,
        "mean_phi": float(ties["phi"].mean()) if n_ties else np.nan,
        "source_edges": len(edges),
    })
    print(f"{market}: {len(edges):,} source edges; {ties['rx_product'].nunique() if not ties.empty else 0} Rx products; {n_ties} top-neighbor rows")

rx_neighbor_ties = pd.concat(all_ties, ignore_index=True) if all_ties else pd.DataFrame()
rx_product_summary = pd.concat(all_product_summaries, ignore_index=True) if all_product_summaries else pd.DataFrame()
rx_neighbor_category_summary = pd.concat(all_category_summaries, ignore_index=True) if all_category_summaries else pd.DataFrame()
rx_nonrx_hubs = pd.concat(all_hubs, ignore_index=True) if all_hubs else pd.DataFrame()
rx_market_summary = pd.DataFrame(market_rows)

paths = {
    "ties": TAB / f"{PREFIX}_ties.csv",
    "product_summary": TAB / f"{PREFIX}_product_summary.csv",
    "category_summary": TAB / f"{PREFIX}_neighbor_category_summary.csv",
    "nonrx_hubs": TAB / f"{PREFIX}_nonrx_hubs.csv",
    "market_summary": TAB / f"{PREFIX}_market_summary.csv",
}
rx_neighbor_ties.to_csv(paths["ties"], index=False)
rx_product_summary.to_csv(paths["product_summary"], index=False)
rx_neighbor_category_summary.to_csv(paths["category_summary"], index=False)
rx_nonrx_hubs.to_csv(paths["nonrx_hubs"], index=False)
rx_market_summary.to_csv(paths["market_summary"], index=False)
for label, path in paths.items():
    print(f"Saved {label}: {path}")


## Quick Views


In [ ]:
print("Market summary")
display(rx_market_summary)

print("Neighbor Category 2 summary")
display(rx_neighbor_category_summary.head(30))

print("Top prescription products by reviews with neighborhood composition")
display(rx_product_summary.groupby("market", group_keys=False).head(10))

print("Top non-prescription hubs connected to prescription products")
display(rx_nonrx_hubs.groupby("market", group_keys=False).head(10))

print("Example tie rows")
display(rx_neighbor_ties.head(30))
